# 03 - Adaptive Topology V0

Plan section 104: information-guided topology evolution versus exactly matched random churn - the Gate 3 question at notebook fidelity.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Adaptive topology vs matched random churn (plan section 104)

The sole question of the third notebook: does information-guided
topology evolution beat exactly the same amount of random topology
evolution? The gate-study machinery pairs seeds, enforces matched
mutation budgets (it aborts otherwise), and emits the objective
verdict - the exact one-tailed binomial sign test used by every
factor-map study.

This notebook runs 4 seeds so it finishes in minutes - far below the
384-seed powered studies in the factor map, so its verdict is a
demo of the machinery, not a Gate 3 claim. The versioned studies
live in `artifacts/study-summaries/`; the same CLI reproduces them
(`PYTHONPATH=src python -m act.evolution.cli --seeds ...`).


In [ ]:
from act.evolution.experiment import ExperimentConfig
from act.evolution.study import run_gate_study

config = ExperimentConfig(
    steps=48,
    mutation_interval=12,
    mutation_budget=3,
    adaptive_policy='probe_loss',
    probe_candidate_cap=4,
    probe_batch_count=2,
    arms='random,adaptive',
    device='cpu',
)
seeds = (1, 2, 3, 42)
study = run_gate_study(config, seeds=seeds, treatment='adaptive')

In [ ]:
print('wins:', study.adaptive_wins, '/', len(study.seeds))
print('mean delta:', round(study.mean_adaptive_delta, 6))
print('matched budgets:', study.matched_mutation_budget)
print('min wins to pass at alpha=0.05:', study.gate_min_wins)
print('binomial p:', study.gate_p_value)
print('gate passed:', study.gate_passed,
      '(4 seeds - a demo verdict, not the powered 384-seed study)')

In [ ]:
# Persist the verdict exactly as every gate study records it.
from act.evolution.registry import append_study

study_record = append_study(
    'artifacts/notebook-records/notebook-gate-study.jsonl', study
)
print('study', study_record['study_id'], '-> gate_passed:',
      study_record['gate_passed'])

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '03_adaptive_topology_v0', 'seeds': list(seeds),
 'wins': study.adaptive_wins,
 'mean_delta': study.mean_adaptive_delta,
 'gate_min_wins': study.gate_min_wins,
 'gate_p_value': study.gate_p_value,
 'gate_passed': study.gate_passed,
 'study_id': study_record['study_id']}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '03-adaptive-v0.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)